In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import copy
from tensordict import TensorDict

# 将项目根目录添加到 Python 路径
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    print(f"Adding project root to Python path: {project_root}")
    sys.path.insert(0, project_root)

# 导入生成器和环境
from implement.generator import MTHFVRPGenerator
from implement.environment import MTHFVRPEnv

## 2. 特征提取说明 (Feature Extraction)

为了供神经网络模型（如 Transformer）使用，环境提供了特征提取方法。

### 2.1 全局特征 (`get_global_features`)
用于 Encoder 输入，包含整个图的静态信息和当前问题的整体属性。

*   **Result (state_feature)**:
    *   `problem_feature`: `[B, H_prob]`。包含 Capacity, Open, TW, Limit, Backhaul, HF 等标志。
    *   `depot_features`: `[B, 1, H_depot]`。Depot 的坐标、时间窗等。
    *   `node_features`: `[B, N, H_node]`。客户的位置、需求、时间窗、服务时间等。
    *   `vehicle_features`: `[B, V, H_veh]`。车辆的容量、成本、数量信息。

### 2.2 当前状态特征 (`get_current_feature_and_mask`)
用于 Decoder 输入，提供决策所需的动态上下文。

*   **Result (current_features)**: `[B, D_curr]`
    *   包括：当前节点索引、当前车辆类型、当前车辆剩余数量、剩余线路线/回程容量、当前累积时间、当前累积距离。
    *   同时返回 `illegal_actions_mask` 供模型 Mask Attention 使用。

## 3. 环境仿真演示

下面我们将初始化一个环境，并使用**随机策略**运行一个完整的 Episode，直到所有实例完成。
如果是训练过程，这里会替换为模型的策略网络采样。

In [ ]:
def run_random_simulation(env, max_steps=1000):
   """
   在环境中运行随机策略仿真。
   返回最终状态和完整的动作历史。
   """
   state = env.reset()
   done = False
   
   # 记录动作历史，用于后续重建路径
   # actions_history: List[Tensor[Batch]]
   actions_history = []
   rewards_history = []
   
   step = 0
   while not done and step < max_steps:
       mask = state["legal_action_mask"] # [B, ActionSpace]
       
       # === 随机策略 ===
       # 简单的 Masked Random Sampling
       # 将非法动作的概率设为 0 (logits = -inf)
       logits = torch.zeros_like(mask, dtype=torch.float)
       logits[~mask] = -float('inf')
       
       # 采样
       probs = torch.softmax(logits, dim=-1)
       dist = torch.distributions.Categorical(probs)
       action = dist.sample()
       
       # 记录
       actions_history.append(action)
       
       # Step
       state, reward, is_done = env.step(state, action)
       rewards_history.append(reward)
       
       done = is_done.all().item()
       step += 1
       
   print(f"Simulation finished in {step} steps.")
   return state, actions_history, rewards_history

# === 初始化配置 ===
NODE_NUM = 20
VEHICLE_NUM = 5
input_kwargs = {
    "num_loc": NODE_NUM, 
    "vehicle_num": VEHICLE_NUM, 
    "variant_preset": "hfvrptw",  # 尝试一个异构车队变体
    "random_seed": 1234
}

# 1. 创建环境 (Batch Size = 1 便于演示)
generator = MTHFVRPGenerator(**input_kwargs)
env = MTHFVRPEnv(generator=generator, batch_size=[1])

# 2. 运行仿真
final_state, actions_hist, rewards_hist = run_random_simulation(env)

In [ ]:
def decode_routes(actions_history, final_state, env, batch_idx=0):
    """
    将动作序列解码为路径列表，并检查解的可行性。
    """
    # 提取特定 batch 的动作序列
    actions = [a[batch_idx].item() for a in actions_history]
    
    # 获取环境参数    
    current_state_locs = final_state["locs"][batch_idx] # [N+1, 2]
    num_vehicle_types = final_state["vehicle_capacity"].shape[1]
    num_nodes = current_state_locs.shape[0] - 1  # 客户数量
    
    routes = []
    current_route = None
    visited_nodes = set()
    
    # print(f"解析动作序列 (Batch {batch_idx}): {actions}")
    
    for action in actions:
        if action == 0:
            # === Return to Depot ===
            if current_route is not None:
                current_route["nodes"].append(0) # 物理上回到 0
                current_route["coords"].append(current_state_locs[0].tolist())
                routes.append(current_route)
                current_route = None # 路径结束
                
        elif 1 <= action <= num_vehicle_types:
            # === Select Vehicle (New Route) ===
            # 如果之前的路径由于某种原因没闭合（例如 Open Route），先保存
            if current_route is not None:
                routes.append(current_route)
            
            vehicle_type_idx = action - 1
            current_route = {
                "vehicle_type": vehicle_type_idx,
                "nodes": [0],  # Start from Depot
                "coords": [current_state_locs[0].tolist()]
            }
            
        else: # action > num_vehicle_types
            # === Visit Customer ===
            customer_idx = action - num_vehicle_types
            if current_route is None:
                print(f"Warning: Visit customer {customer_idx} without vehicle selected! (Action: {action})")
                continue
                
            current_route["nodes"].append(customer_idx)
            current_route["coords"].append(current_state_locs[customer_idx].tolist())
            visited_nodes.add(customer_idx)
            
    # 处理最后一条路径（如果是 Open Route 可能不以 0 结尾）
    if current_route is not None:
        routes.append(current_route)

    # === 检查可行性 (Infeasible Check) ===
    # 注意：我们不能直接使用 final_state["visited"] 来判断，因为 Environment 在 step 函数中
    # 处理 Infeasible Done 时，会强制将 visited 全置为 True并计算惩罚成本 以便结束 Episode。
    # 因此，我们必须根据 action 历史中实际访问过的节点来判断。
    
    all_customers = set(range(1, num_nodes + 1))
    unvisited_nodes = list(sorted(all_customers - visited_nodes))
    is_feasible = len(unvisited_nodes) == 0
    
    if not is_feasible:
        print(f"\n[Warning] Solution is INFEASIBLE!")
        print(f"Unvisited Nodes ({len(unvisited_nodes)}/{num_nodes}): {unvisited_nodes}")
    else:
        print(f"\n[Info] Solution is FEASIBLE (All {num_nodes} customers visited).")
        
    return routes, current_state_locs, is_feasible, unvisited_nodes

# 执行解析
routes, locs, is_feasible, unvisited = decode_routes(actions_hist, final_state, env, batch_idx=0)

# 打印结果
print(f"\n=== 解析结果 (共 {len(routes)} 条路径) ===")
for i, route in enumerate(routes):
    node_seq = " -> ".join(map(str, route['nodes']))
    print(f"Route {i+1} (Vehicle Type {route['vehicle_type']}): {node_seq}")

In [ ]:
def plot_vrp_solution(locs, routes, title="VRP Solution Visualization"):
    """
    可视化 VRP 路径解
    """
    plt.figure(figsize=(10, 8))
    
    # 1. 绘制节点
    # Depot (Index 0)
    plt.scatter(locs[0, 0], locs[0, 1], c='red', marker='s', s=200, label='Depot', zorder=10)
    # Customers (Index 1+)
    plt.scatter(locs[1:, 0], locs[1:, 1], c='blue', alpha=0.6, s=50, label='Customer')
    
    # 标记客户 ID
    for i in range(1, len(locs)):
        plt.annotate(str(i), (locs[i, 0], locs[i, 1]), xytext=(0, 5), textcoords='offset points', ha='center')

    # 2. 绘制路径
    # 使用不同的颜色区分路径
    cmap = plt.get_cmap("tab10")
    
    for i, route in enumerate(routes):
        coords = np.array(route['coords'])
        color = cmap(i % 10)
        v_type = route['vehicle_type']
        
        # 画线
        plt.plot(coords[:, 0], coords[:, 1], c=color, linewidth=2, label=f"Route {i+1} (Type {v_type})")
        
        # 画箭头 (表示方向)
        # 在每段中间画一个小箭头
        for j in range(len(coords) - 1):
            start = coords[j]
            end = coords[j+1]
            plt.arrow(start[0], start[1], 
                      (end[0]-start[0])*0.5, (end[1]-start[1])*0.5, # 箭头位置在中间
                      head_width=0.02, head_length=0.03, fc=color, ec=color, length_includes_head=True)

    plt.title(title)
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# 这里的 locs 是 Tensor，需要转 numpy
locs_np = locs.cpu().numpy()
plot_vrp_solution(locs_np, routes, title=f"MTHFVRP Solution ({input_kwargs['variant_preset']})")